# SVM no lineal con kernel RBF — Clasificación de flores Iris y círculos concéntricos

**Resultados de aprendizaje:** (1) justificar un kernel no lineal; (2) interpretar $C$ y $\gamma$; (3) distinguir entrenamiento, validación y prueba; (4) visualizar vectores de soporte y fronteras; (5) comparar RBF con SVM lineal.

**Ruta:** 1. Fundamentos → 2. Datos Iris → 3. Preparación → 4. RBF → 5. Evaluación → 6. Fronteras → 7. Hiperparámetros → 8. Validación cruzada → 9. Comparación → 10. Círculos → 11. Actividades.


## 1. Fundamento conceptual: ¿por qué RBF?
Un SVM lineal construye una frontera $w^Tx+b=0$. Cuando la separación requiere curvas, un kernel permite obtener decisiones no lineales sin calcular explícitamente el mapeo a otro espacio.

**Kernel gaussiano RBF:**
$$K(x_i,x_j)=\exp(-\gamma\lVert x_i-x_j\rVert^2).$$
- **$\gamma$ bajo:** similitud decrece lentamente con la distancia; fronteras habitualmente suaves.
- **$\gamma$ alto:** similitud localizada; fronteras potencialmente más complejas.
- **$C$ bajo:** regularización fuerte; tolera más violaciones del margen.
- **$C$ alto:** penaliza más las violaciones; puede sobreajustar.

**Precisión:** RBF no equivale a trasladar los puntos a exactamente tres dimensiones. En general, su espacio de características es infinito-dimensional. Las conclusiones sobre sobreajuste deben basarse en resultados de validación.

## 2. Librerías y reproducibilidad

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, make_circles
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
RANDOM_STATE = 42
plt.rcParams['figure.figsize'] = (8, 5)
print('Entorno preparado')

## 3. Iris: explorar las dos características de sépalos
Se usan **longitud y anchura del sépalo** para poder dibujar fronteras en dos dimensiones. Las otras dos variables (pétalos) se reservan para una extensión.

In [ ]:
iris = load_iris()
X = iris.data[:, :2]
y = iris.target
class_names = iris.target_names
print('Forma X:', X.shape, '| clases:', dict(zip(class_names, np.bincount(y))))
fig, ax = plt.subplots()
for k, name in enumerate(class_names):
    ax.scatter(X[y==k,0], X[y==k,1], label=name, alpha=.75, edgecolor='k', linewidth=.3)
ax.set(xlabel='Longitud del sépalo (cm)', ylabel='Anchura del sépalo (cm)', title='Iris: solapamiento entre especies')
ax.legend(); plt.show()

## 4. Partición estratificada y estandarización
**70 % entrenamiento y 30 % prueba.** La particion preserva las proporciones de clases (stratify=y). El escalador aprende media y desviación **solo de entrenamiento (train)** para evitar fuga de información. Las estadísticas de prueba (test) no tienen por qué tener media cero.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.30, random_state=RANDOM_STATE, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('Entrenamiento:', X_train.shape, np.bincount(y_train))
print('Prueba:', X_test.shape, np.bincount(y_test))
print('Medias originales (train):', np.round(scaler.mean_, 3))
print('Medias escaladas (train):', np.round(X_train_scaled.mean(axis=0), 6))
print('Desviaciones escaladas:', np.round(X_train_scaled.std(axis=0), 6))

## 5. Entrenar SVM no lineal RBF
Se toman inicialmente los siguientes valores: `C=1` y `gamma="auto"`. Con dos variables, `auto` equivale a $1/2=0.5$. `gamma="scale"` es otra opción y no debe confundirse con `auto`.

In [ ]:
model = SVC(kernel='rbf', C=1.0, gamma='auto')
model.fit(X_train_scaled, y_train)
print('Gamma efectivo:', model._gamma)
print('Vectores de soporte por clase:', dict(zip(class_names, model.n_support_)))
print('Total de vectores de soporte únicos:', len(model.support_))
print('Clasificadores binarios OvO:', len(class_names)*(len(class_names)-1)//2)

### Interpretación del entrenamiento
`SVC` resuelve internamente tres comparaciones **uno contra uno (OvO)**: Setosa–Versicolor, Setosa–Virginica y Versicolor–Virginica. `n_support_` cuenta vectores de soporte **únicos por clase**; no es la suma por cada problema binario. Un vector de soporte puede estar dentro del margen o mal clasificado.

## 6. Predicción y evaluación
La evaluación en prueba se realiza **una vez** para el modelo base usando la métrica Accuracy.

In [ ]:
y_pred = model.predict(X_test_scaled)
print('Primeras 10 predicciones:', y_pred[:10])
print('Primeras 10 etiquetas:  ', y_test[:10])
print('Accuracy entrenamiento:', round(model.score(X_train_scaled, y_train), 4))
print('Accuracy prueba:', round(accuracy_score(y_test, y_pred), 4))

## 7. Fronteras de decisión y vectores de soporte
La malla se crea en **centímetros**, pero antes de predecir se transforma mediante el mismo escalador. Los vectores de soporte se devuelven a centímetros con `inverse_transform`. Círculos: entrenamiento; triángulos: prueba; anillos negros: vectores de soporte.

In [ ]:
def plot_boundary(ax, clf, scaler, Xtr, ytr, Xte, yte, title, show_support=True):
    Xall = np.vstack([Xtr, Xte])
    xlo, xhi = Xall[:,0].min()-.5, Xall[:,0].max()+.5
    ylo, yhi = Xall[:,1].min()-.5, Xall[:,1].max()+.5
    xx, yy = np.meshgrid(np.linspace(xlo,xhi,240), np.linspace(ylo,yhi,240))
    mesh = np.c_[xx.ravel(), yy.ravel()]
    Z = clf.predict(scaler.transform(mesh)).reshape(xx.shape)
    ax.contourf(xx,yy,Z,levels=np.arange(-.5,len(np.unique(np.r_[ytr,yte]))+.5,1),alpha=.17,cmap='viridis')
    ax.scatter(Xtr[:,0],Xtr[:,1],c=ytr,cmap='viridis',vmin=0,vmax=2,s=35,edgecolors='k',linewidths=.35,label='Entrenamiento')
    ax.scatter(Xte[:,0],Xte[:,1],c=yte,cmap='viridis',vmin=0,vmax=2,s=55,marker='^',edgecolors='k',linewidths=.4,label='Prueba')
    if show_support and hasattr(clf,'support_vectors_'):
        sv = scaler.inverse_transform(clf.support_vectors_)
        ax.scatter(sv[:,0],sv[:,1],s=125,facecolors='none',edgecolors='black',linewidths=1.3,label='Vectores de soporte')
    ax.set(xlabel='Longitud del sépalo (cm)',ylabel='Anchura del sépalo (cm)',title=title)
    return ax
fig, ax = plt.subplots(figsize=(9,6))
plot_boundary(ax,model,scaler,X_train,y_train,X_test,y_test,
              f'RBF | C=1, gamma={model._gamma:g} | Accuracy prueba={model.score(X_test_scaled,y_test):.3f}')
ax.legend(loc='best'); plt.tight_layout(); plt.show()

## 8. Experimentos controlados con gamma y C
**Regla:** cambiar solo un hiperparámetro a la vez. Comparar Accuracy en entrenamiento, prueba y cantidad de vectores de soporte. Los resultados de prueba aquí son **ilustrativos**; no seleccionar hiperparámetros mirando repetidamente la prueba. Para selección formal usar la validación cruzada de la sección 9.

In [ ]:
def experiment_table(param_name, values, fixed):
    rows=[]
    for v in values:
        kw = dict(kernel='rbf', **fixed, **{param_name:v})
        clf=SVC(**kw).fit(X_train_scaled,y_train)
        rows.append({param_name:v, 'Accuracy train':clf.score(X_train_scaled,y_train),
                     'Accuracy test (solo ilustrativo)':clf.score(X_test_scaled,y_test),
                     'N vectores soporte':len(clf.support_)})
    return pd.DataFrame(rows)
gamma_values=[.01,.1,.5,1,10]
print('Experimento gamma (C=1):')
display(experiment_table('gamma',gamma_values,{'C':1.0}).round(3))
C_values=[.01,.1,1,10,100]
print('Experimento C (gamma=0.5):')
display(experiment_table('C',C_values,{'gamma':.5}).round(3))

In [ ]:
fig,axes=plt.subplots(1,5,figsize=(19,3.8),sharex=True,sharey=True)
for ax,g in zip(axes,gamma_values):
    clf=SVC(kernel='rbf',C=1,gamma=g).fit(X_train_scaled,y_train)
    plot_boundary(ax,clf,scaler,X_train,y_train,X_test,y_test,f'gamma={g}',False)
fig.suptitle('Fronteras RBF: variación de gamma con C=1'); plt.tight_layout(); plt.show()
fig,axes=plt.subplots(1,5,figsize=(19,3.8),sharex=True,sharey=True)
for ax,c in zip(axes,C_values):
    clf=SVC(kernel='rbf',C=c,gamma=.5).fit(X_train_scaled,y_train)
    plot_boundary(ax,clf,scaler,X_train,y_train,X_test,y_test,f'C={c}',False)
fig.suptitle('Fronteras RBF: variación de C con gamma=0.5'); plt.tight_layout(); plt.show()

**Interpretación de gráficas**
1. Primera línea: efecto de gamma manteniendo C=1

Gamma determina qué tan rápidamente disminuye la similitud entre dos muestras a medida que aumenta su distancia.

Interpretación de las cinco gráficas:
- γ = 0,01	Influencia muy amplia. Fronteras suaves, con escasa capacidad para adaptarse a detalles locales.
- γ = 0,1	Mayor capacidad para adaptarse a la distribución, manteniendo fronteras relativamente suaves.
- γ = 0,5	Influencia intermedia. Corresponde al valor efectivo del modelo original con gamma='auto'.
- γ = 1	Influencia más localizada. Pueden observarse curvaturas más pronunciadas.
- γ = 10	Influencia muy localizada. Pueden aparecer regiones pequeñas y fronteras complejas alrededor de grupos de muestras.

Al aumentar gamma, la influencia espacial de cada muestra se vuelve más localizada. Esto permite construir fronteras potencialmente más complejas, pero no garantiza que sean mejores.

Un gamma muy alto puede generar sobreajuste, aunque es necesario contrastar Accuracy de entrenamiento y validación para comprobarlo.

2. Segunda línea: efecto de C manteniendo gamma=0,5

El parámetro C controla cuánto penaliza SVM las violaciones del margen durante el entrenamiento.

Interpretación de las cinco gráficas:
- C = 0,01	Regularización muy fuerte. El modelo prioriza un margen amplio y tolera más violaciones.
- C = 0,1	Regularización fuerte. Las fronteras siguen siendo relativamente simples.
- C = 1	Compromiso intermedio. Es la configuración de referencia del cuaderno.
- C = 10	Penalización mayor de las violaciones. La frontera puede adaptarse más a las muestras.
- C = 100	Regularización débil. Se busca reducir las violaciones del margen, con mayor riesgo potencial de sobreajuste.

Diferencia esencial: aumentar C no significa aumentar directamente la curvatura. Significa cambiar el equilibrio entre maximizar el margen y penalizar las violaciones. La geometría resultante depende también de gamma y de los datos.

**Conclusiones**

- Gamma controla el alcance de la similitud entre observaciones; C controla cuánto se penalizan las violaciones del margen.
- No se debe seleccionar una frontera porque visualmente parezca mejor ni porque obtenga el mayor Accuracy en una única partición de prueba. La selección de hiperparámetros debe realizarse mediante validación cruzada, como se desarrolla en el numeral 9

## 9. Selección de hiperparámetros con validación cruzada
El `Pipeline` ajusta el escalador **dentro de cada fold**, evitando fuga de información. La búsqueda usa solo entrenamiento. Se evalúa el mejor modelo en prueba **después** de seleccionar hiperparámetros.

In [ ]:
pipe=Pipeline([('scale',StandardScaler()),('svm',SVC(kernel='rbf'))])
param_grid={'svm__C':[.01,.1,1,10,100], 'svm__gamma':[.01,.1,.5,1,10]}
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)
search=GridSearchCV(pipe,param_grid,cv=cv,scoring='accuracy',n_jobs=-1,refit=True)
search.fit(X_train,y_train)
print('Mejores hiperparámetros:',search.best_params_)
print('Accuracy media CV:',round(search.best_score_,4))
print('Accuracy prueba (evaluación final):',round(search.score(X_test,y_test),4))
results=pd.DataFrame(search.cv_results_)
display(results[['param_svm__C','param_svm__gamma','mean_test_score','std_test_score']]
        .sort_values('mean_test_score',ascending=False).head(10).round(3))

## 10. Comparación justa: SVM lineal frente a RBF
Ambos modelos utilizan las **mismas particiones y estandarización**. El SVM RBF base y el lineal base emplean `C=1`; no se debe afirmar que uno es mejor por una sola partición.

In [ ]:
models={'Lineal C=1':SVC(kernel='linear',C=1),
        'RBF C=1 gamma=auto':SVC(kernel='rbf',C=1,gamma='auto')}
rows=[]
for name,clf in models.items():
    clf.fit(X_train_scaled,y_train)
    rows.append({'Modelo':name,'Accuracy train':clf.score(X_train_scaled,y_train),
                 'Accuracy test':clf.score(X_test_scaled,y_test),
                 'Vectores soporte':len(clf.support_)})
display(pd.DataFrame(rows).round(4))
fig,axs=plt.subplots(1,2,figsize=(13,5))
for ax,(name,clf) in zip(axs,models.items()):
    plot_boundary(ax,clf,scaler,X_train,y_train,X_test,y_test,name,True)
axs[1].legend(loc='best',fontsize=8)
plt.tight_layout();plt.show()

## 11. Caso sintético: círculos concéntricos
Iris no garantiza una ventaja del kernel RBF. En `make_circles`, las clases forman anillos: una recta no puede separar perfectamente un círculo interior de otro exterior. Aquí se muestra claramente el motivo de usar un kernel no lineal.

In [ ]:
Xc,yc=make_circles(n_samples=400,factor=.4,noise=.07,random_state=RANDOM_STATE)
Xctr,Xcte,yctr,ycte=train_test_split(Xc,yc,test_size=.3,stratify=yc,random_state=RANDOM_STATE)
sc=StandardScaler().fit(Xctr)
Xctr_s=sc.transform(Xctr);Xcte_s=sc.transform(Xcte)
fig,axs=plt.subplots(1,2,figsize=(12,5))
for ax,(name,clf) in zip(axs,[('Lineal',SVC(kernel='linear',C=1)),('RBF',SVC(kernel='rbf',C=1,gamma='scale'))]):
    clf.fit(Xctr_s,yctr)
    xx,yy=np.meshgrid(np.linspace(-1.5,1.5,250),np.linspace(-1.5,1.5,250))
    zz=clf.predict(sc.transform(np.c_[xx.ravel(),yy.ravel()])).reshape(xx.shape)
    ax.contourf(xx,yy,zz,alpha=.18,cmap='coolwarm')
    ax.scatter(Xctr[:,0],Xctr[:,1],c=yctr,cmap='coolwarm',s=20,edgecolors='k',linewidths=.2,label='Train')
    ax.scatter(Xcte[:,0],Xcte[:,1],c=ycte,cmap='coolwarm',marker='^',s=35,edgecolors='k',linewidths=.3,label='Test')
    ax.set(title=f'{name}: Accuracy test={clf.score(Xcte_s,ycte):.3f}',xlabel='Característica 1',ylabel='Característica 2',aspect='equal')
axs[1].legend();plt.tight_layout();plt.show()

## 12. Actividad de ampliación: sépalos frente a pétalos
**Reto:** repetir la evaluación con `X = iris.data[:, 2:4]` (longitud y anchura de pétalos), preservando `stratify`, la partición y la separación del escalamiento. Comparar con los resultados obtenidos para sépalos. **No** reutilizar un escalador entrenado con otras columnas.

## 13. Preguntas de reflexión
1. ¿Por qué una recta no separa adecuadamente los círculos concéntricos?
2. ¿Qué representa $K(x_i,x_j)$ en el kernel RBF?
3. ¿Cómo cambia la frontera cuando aumenta gamma, manteniendo C fijo?
4. ¿Qué diferencia existe entre aumentar C y aumentar gamma?
5. ¿Por qué RBF no tiene que superar al lineal en Iris?
6. ¿Qué representa `n_support_` en un problema multiclase?
7. ¿Por qué el escalamiento debe estar dentro del Pipeline en validación cruzada?
8. ¿Qué conclusiones cambian cuando se utilizan pétalos en lugar de sépalos?

**Cierre:** un modelo flexible no es necesariamente un mejor predictor; la selección debe basarse en validación y evaluación independiente.